# 🔧 Project 7.3 — Machine Maintenance Scheduler
**DSA for Mechatronics · Week 07 — Heaps & Priority Queues**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq, math
from collections import Counter, defaultdict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A maintenance team must service machines in a factory.
Each service job has a **start time** and **end time**.
We need to answer two questions:
1. **How many technicians** are needed to handle all jobs with no overlap? (Meeting Rooms II)
2. **What is the minimum idle time** if we assign jobs to a fixed crew greedily?

A **min-heap of end times** solves both:
- Sort jobs by start time
- If the earliest-finishing technician is free when a new job starts → reuse them (heap pop + push)
- Otherwise → assign a new technician (heap push only)


## Step 1 — Generate your maintenance job schedule

In [ ]:
# ── Personalised parameters ───────────────────────────────────────
N_JOBS       = random.randint(14, 24)
N_MACHINES   = random.randint(6, 12)
DAY_HOURS    = 480   # 8-hour shift in minutes
MAX_DURATION = random.randint(40, 90)
CREW_SIZE    = random.randint(3, 5)

MACHINE_NAMES = [f"MCH-{i+1:02d}" for i in range(N_MACHINES)]
JOB_TYPES     = ["LUBRICATION","CALIBRATION","FILTER_CHANGE","BELT_CHECK",
                 "ELECTRICAL","COOLANT","ENCODER_CHECK","SAFETY_TEST","FIRMWARE"]

jobs = []
for i in range(N_JOBS):
    start    = random.randint(0, DAY_HOURS - MAX_DURATION)
    duration = random.randint(15, MAX_DURATION)
    end      = start + duration
    machine  = random.choice(MACHINE_NAMES)
    jtype    = random.choice(JOB_TYPES)
    jobs.append({"id": i+1, "machine": machine, "type": jtype,
                 "start": start, "end": end, "duration": duration})

jobs.sort(key=lambda j: j["start"])

def fmt_time(minutes):
    """Convert minutes from shift start to HH:MM."""
    h, m = divmod(minutes, 60)
    return f"{8+h:02d}:{m:02d}"

print(f"Maintenance schedule parameters:")
print(f"  Jobs          : {N_JOBS}")
print(f"  Machines      : {N_MACHINES}")
print(f"  Crew size     : {CREW_SIZE} technicians")
print(f"  Shift         : 08:00 – 16:00")
print()
print(f"  {'ID':>3}  {'Machine':<10}  {'Type':<16}  {'Start':>8}  {'End':>8}  {'Dur':>5}")
print(f"  {'─'*3}  {'─'*10}  {'─'*16}  {'─'*8}  {'─'*8}  {'─'*5}")
for j in jobs:
    print(f"  {j['id']:3d}  {j['machine']:<10}  {j['type']:<16}  "
          f"{fmt_time(j['start']):>8}  {fmt_time(j['end']):>8}  {j['duration']:5d}m")


## Step 2 — Min-heap: minimum technicians needed (Meeting Rooms II)

In [ ]:
def min_technicians(jobs):
    """
    Find the minimum number of technicians needed to service all jobs.

    Algorithm (Meeting Rooms II):
      1. Sort jobs by start time.
      2. Use a min-heap of END times (one entry per active technician).
      3. For each job:
         - If heap top (earliest end) <= job start → that technician is free.
           Pop and push new end time (same technician, new job).
         - Else → all technicians are busy; add a new one (push end time).
      4. Heap size at the end = minimum technicians needed.

    O(n log n) time.
    """
    if not jobs: return 0, []
    end_heap = []   # min-heap of end times
    log = []
    for j in jobs:
        if end_heap and end_heap[0] <= j["start"]:
            freed = heapq.heapreplace(end_heap, j["end"])
            log.append((j["id"], "REUSE", freed, j["end"]))
        else:
            heapq.heappush(end_heap, j["end"])
            log.append((j["id"], "NEW", None, j["end"]))
    return len(end_heap), log

min_crew, assign_log = min_technicians(jobs)

print(f"Meeting Rooms II — minimum crew size:")
print(f"  Jobs        : {N_JOBS}")
print(f"  Min needed  : {min_crew} technician(s)")
print()
print(f"  {'Job':>4}  {'Action':<8}  {'Freed at':>10}  {'New end':>10}")
print(f"  {'─'*4}  {'─'*8}  {'─'*10}  {'─'*10}")
for jid, action, freed, new_end in assign_log:
    freed_str = fmt_time(freed) if freed else "—"
    print(f"  {jid:4d}  {action:<8}  {freed_str:>10}  {fmt_time(new_end):>10}")


## Step 3 — Greedy crew assignment and idle time

In [ ]:
def assign_to_crew(jobs, crew_size):
    """
    Assign jobs to a fixed crew greedily:
    Always give the next job to the technician who finishes earliest.
    Returns per-technician workload and total idle time.
    """
    # heap: (current_end_time, tech_id)
    crew_heap = [(0, i+1) for i in range(crew_size)]
    heapq.heapify(crew_heap)

    assignments = []   # (tech_id, job_id, start, end, idle_gap)
    tech_workload = defaultdict(int)

    for j in jobs:
        earliest_end, tech_id = heapq.heappop(crew_heap)
        idle = max(0, j["start"] - earliest_end)
        tech_workload[tech_id] += j["duration"]
        assignments.append({
            "tech": tech_id, "job": j["id"],
            "start": j["start"], "end": j["end"],
            "idle": idle
        })
        heapq.heappush(crew_heap, (j["end"], tech_id))

    return assignments, tech_workload, crew_heap

assignments, workloads, final_heap = assign_to_crew(jobs, CREW_SIZE)
total_idle = sum(a["idle"] for a in assignments)
max_end    = max(a["end"] for a in assignments)

print(f"Greedy assignment to {CREW_SIZE}-person crew:")
print(f"  {'Tech':>5}  {'Job':>4}  {'Start':>8}  {'End':>8}  {'Idle gap':>9}")
print(f"  {'─'*5}  {'─'*4}  {'─'*8}  {'─'*8}  {'─'*9}")
for a in assignments:
    print(f"  T{a['tech']:4d}  {a['job']:4d}  {fmt_time(a['start']):>8}  "
          f"{fmt_time(a['end']):>8}  {a['idle']:6d} m")

print(f"\n  Total idle time   : {total_idle} min")
print(f"  Shift end (last)  : {fmt_time(max_end)}")
print()
print(f"  Workload per technician:")
for tech in sorted(workloads):
    bar = "█" * (workloads[tech] // 10)
    print(f"    T{tech}: {workloads[tech]:4d} min  {bar}")


## Step 4 — Task reorganise (greedy: most-frequent first)

In [ ]:
def reorganize_jobs(jobs):
    """
    Reorganise job type sequence so no two consecutive jobs are the same type.
    Uses a max-heap of (count, type) — always schedule the most-frequent type next.
    Returns reorganised list or None if impossible.
    """
    type_counts = Counter(j["type"] for j in jobs)
    max_count   = max(type_counts.values())
    if max_count > (len(jobs) + 1) // 2:
        return None   # impossible

    max_heap = [(-cnt, jtype) for jtype, cnt in type_counts.items()]
    heapq.heapify(max_heap)

    result = []
    while len(max_heap) >= 2:
        cnt1, t1 = heapq.heappop(max_heap)
        cnt2, t2 = heapq.heappop(max_heap)
        result.extend([t1, t2])
        if cnt1 + 1 < 0: heapq.heappush(max_heap, (cnt1 + 1, t1))
        if cnt2 + 1 < 0: heapq.heappush(max_heap, (cnt2 + 1, t2))
    if max_heap:
        result.append(max_heap[0][1])

    # Verify no two consecutive same
    valid = all(result[i] != result[i+1] for i in range(len(result)-1))
    return result, valid

reorg_result = reorganize_jobs(jobs)

type_counts = Counter(j["type"] for j in jobs)
print("Job type frequency:")
for jtype, cnt in type_counts.most_common():
    bar = "█" * cnt
    print(f"  {jtype:<18}: {bar} ({cnt})")

print()
if reorg_result:
    seq, valid = reorg_result
    print(f"Reorganised sequence (no consecutive same type):")
    print(f"  {seq}")
    print(f"  Valid (no adjacent duplicates): {'✅ YES' if valid else '❌ NO'}")
else:
    print("  Reorganisation IMPOSSIBLE (one type dominates > half)")
    valid = False


## 📋 Final Report

In [ ]:
W = 56
print("╔" + "═"*W + "╗")
print("║" + " MACHINE MAINTENANCE SCHEDULER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  SCHEDULE PARAMETERS" + " "*(W-21) + "║")
print(f"║  {'Jobs':<26}: {N_JOBS:<26}║")
print(f"║  {'Machines':<26}: {N_MACHINES:<26}║")
print(f"║  {'Crew size (fixed)':<26}: {CREW_SIZE:<26}║")
print("╠" + "═"*W + "╣")
print("║  MEETING ROOMS II" + " "*(W-18) + "║")
print(f"║  {'Min technicians needed':<26}: {min_crew:<26}║")
new_count  = sum(1 for _,a,_,_ in assign_log if a=="NEW")
reuse_count= sum(1 for _,a,_,_ in assign_log if a=="REUSE")
print(f"║  {'New assignments':<26}: {new_count:<26}║")
print(f"║  {'Technician reuses':<26}: {reuse_count:<26}║")
print("╠" + "═"*W + "╣")
print("║  GREEDY CREW ASSIGNMENT" + " "*(W-24) + "║")
print(f"║  {'Total idle time':<26}: {total_idle} min{'':<21}║")
print(f"║  {'Shift end (latest job)':<26}: {fmt_time(max_end):<26}║")
busiest = max(workloads, key=workloads.get)
print(f"║  {'Busiest technician':<26}: T{busiest} ({workloads[busiest]} min){'':<14}║")
print("╠" + "═"*W + "╣")
print("║  REORGANISE CHECK" + " "*(W-18) + "║")
if reorg_result:
    print(f"║  {'Reorganisation possible':<26}: {'YES ✅':<26}║")
    print(f"║  {'Valid sequence':<26}: {'YES ✅' if valid else 'NO ❌':<26}║")
else:
    print(f"║  {'Reorganisation possible':<26}: {'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which heap concept did you find most important, and why?**
Pick one technique (sift-up, sift-down, two-heap median, heapify…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
